# LungInsight — GPU Inference + Grad-CAM

Runs inference and produces per-head Grad-CAM heatmaps for a single nodule patch on GPU.
Much faster than CPU: ~5 seconds per nodule vs 5-15 minutes locally.

**Before running:**
- GPU runtime must be enabled (Runtime → Change runtime type → T4 GPU)
- `cir_multihead_pipeline.py` and `se_resnet3d.py` uploaded to `/content/LungInsight/`
- Checkpoint and patches accessible on Drive

In [ ]:
%pip install -q torch torchvision numpy pandas pylidc grad-cam

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
if not os.path.isdir(ROOT_DIR):
    raise RuntimeError(
        'Upload cir_multihead_pipeline.py and se_resnet3d.py to /content/LungInsight first.'
    )
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAMPlusPlus
import pytorch_grad_cam.base_cam as pytorch_grad_cam_base_cam
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Monkeypatch for 3D heatmap shape compatibility with pytorch_grad_cam
_orig_scale = pytorch_grad_cam_base_cam.scale_cam_image
def _scale_3d(cam, target_size=None):
    cam = np.asarray(cam)
    if cam.ndim == 5:
        cam = cam.reshape(-1, *cam.shape[2:])
    return _orig_scale(cam, target_size)
pytorch_grad_cam_base_cam.scale_cam_image = _scale_3d

from cir_multihead_pipeline import create_multihead_model, FEATURE_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Heads:', FEATURE_NAMES)

## Config — set nodule and paths here

In [ ]:
# Nodule to run inference on
NODULE_ID = 'LIDC-IDRI-0007_002'  # change this to any nodule_id from your manifest

# Paths — adjust if your Drive layout differs
DRIVE_DIR    = '/content/drive/MyDrive/LungInsight'
PATCHES_DIR  = f'{DRIVE_DIR}/cpu_split/patches'  # where .npy files live after unzipping
CHECKPOINT   = f'{DRIVE_DIR}/best_model_gpu.pth'

PATCH_PATH = f'{PATCHES_DIR}/nodule_{NODULE_ID}.npy'

# Verify
for label, path in [('Patch', PATCH_PATH), ('Checkpoint', CHECKPOINT)]:
    exists = os.path.isfile(path)
    print(f'{label}: {path}  [{"OK" if exists else "NOT FOUND"}]')
    if not exists:
        raise FileNotFoundError(f'{label} not found: {path}')

## Load model and patch

In [ ]:
class HeadOnlyModel(torch.nn.Module):
    """Wraps multi-head dict-output model to expose one head as a plain tensor.
    Required because pytorch_grad_cam expects tensor output, not a dict."""
    def __init__(self, model, head_name):
        super().__init__()
        self.model = model
        self.head_name = head_name

    def forward(self, x):
        out = self.model(x)[self.head_name]
        return out.unsqueeze(1) if out.dim() == 1 else out


# Load patch
patch = np.load(PATCH_PATH)
assert patch.shape == (64, 64, 64), f'Expected (64,64,64), got {patch.shape}'
x = torch.from_numpy(patch.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)

# Load model
model = create_multihead_model(head_names=FEATURE_NAMES, device=device)
state_dict = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded.')

# Forward pass for confidence scores
with torch.no_grad():
    outputs = model(x)

probs = {head: float(outputs[head].cpu().item()) for head in FEATURE_NAMES}
print('\nConfidence scores:')
for head, prob in probs.items():
    print(f'  {head}: {prob:.4f}')

## Grad-CAM per head

In [ ]:
target_layer_module = model.layer4
heatmaps = {}

for head in FEATURE_NAMES:
    wrapped = HeadOnlyModel(model, head)
    wrapped.eval()
    with GradCAMPlusPlus(model=wrapped, target_layers=[target_layer_module]) as cam:
        grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(0)])

    heatmap = grayscale_cam[0] if grayscale_cam.ndim == 4 else grayscale_cam
    if heatmap.shape != (64, 64, 64):
        if heatmap.shape[0] == 1 and heatmap.shape[1:] == (64, 64, 64):
            heatmap = heatmap[0]
        else:
            raise RuntimeError(f'Unexpected heatmap shape for {head}: {heatmap.shape}')
    heatmap = np.clip(
        (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8),
        0.0, 1.0
    )
    heatmaps[head] = heatmap.astype(np.float32)
    print(f'  {head}: done')

print('Grad-CAM complete.')

## Visualization — inline and saved to Drive

In [ ]:
mid = patch.shape[0] // 2  # central axial slice

# Normalize CT for display
ct = patch[mid].astype(np.float32)
p1, p99 = np.percentile(ct, [1, 99])
ct_norm = np.clip((ct - p1) / (p99 - p1 + 1e-8), 0.0, 1.0)

n = len(FEATURE_NAMES)
fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))

for i, head in enumerate(FEATURE_NAMES):
    heatmap_slice = heatmaps[head][mid]
    prob = probs[head]

    # Top row: CT slice with score
    axes[0, i].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'{head}\n{prob:.3f}', fontsize=8)
    axes[0, i].axis('off')

    # Bottom row: heatmap overlay
    axes[1, i].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(heatmap_slice, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CT slice', fontsize=8)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=8)
fig.suptitle(f'nodule_{NODULE_ID}', fontsize=10, fontweight='bold')
plt.tight_layout()

# Save to Drive
out_png = f'{DRIVE_DIR}/nodule_{NODULE_ID}_vis.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
print(f'Saved to Drive: {out_png}')

# Display inline
from IPython.display import display
from PIL import Image
display(Image.open(out_png))